# 07 — Experiment 1.2: class imbalance ablation

Does training a classifier with cost-sensitive learning
(`class_weight='balanced'` for Random Forest, `scale_pos_weight=n0/n1`
for LightGBM) preserve or degrade the XAI explanation quality compared
to training on the imbalanced data as-is? Hypothesis from the
literature: cost-sensitive learning preserves fidelity better than
resampling because it does not modify the feature distribution.

**Protocol:**

1. Train balanced RF and LightGBM on both datasets.
2. Compute SHAP and LIME attributions for those balanced models on the
   same 200 balanced test indices used throughout Modules 1-3.
3. Re-run the metrics from Modules 1 (fidelity), 2 (stability), and 3
   (BRAS) for both modes (Original, Balanced).
4. Compute per-metric deltas and visualize them.

**Coverage:** {RF, LGB} × {SHAP, LIME} × {Elliptic, Ethereum} = 8
configurations × 2 modes = 16 evaluations.

**Prerequisite:** 02a, 02b, 03a, 03b, 04, 05, 06.


In [ ]:
import os
from datetime import datetime

import joblib
import lightgbm as lgb
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

from xai_blockchain_framework.config import PATHS, set_seed
from xai_blockchain_framework.models import make_fraud_predict_fn
from xai_blockchain_framework.xai.shap_wrapper import ShapTreeExplainer
from xai_blockchain_framework.xai.lime_wrapper import LimeTabularExplainer
from xai_blockchain_framework.metrics import (
    comprehensiveness, sufficiency, infidelity,
    lipschitz_stability, rank_stability_kendall, cov_bootstrap, identity_score,
    evaluate_bras,
)
from xai_blockchain_framework.rules import elliptic_rules, ethereum_rules

set_seed(42)

RESULTS_PATH = str(PATHS.results_dir)
FIGURES_PATH = str(PATHS.figures_dir)
MODELS_PATH = str(PATHS.models_dir)
PROCESSED_PATH = str(PATHS.data_processed)
for p in [RESULTS_PATH, FIGURES_PATH, PROCESSED_PATH, MODELS_PATH]:
    os.makedirs(p, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')

N_EVAL = 200
N_EVAL_LIME = 100
N_PERTURBATIONS = 50
SIGMA = 0.1
N_BOOTSTRAP = 5
N_IDENTITY = 5
PERT_SCALE = 0.01
N_COV_SAMPLE = 15
RANDOM_STATE = 42
K_FID = 5
TOP_K_BRAS = 5
ALPHA_BRAS = 0.5

print("=" * 70)
print("EXPERIMENT 1.2  CLASS IMBALANCE ABLATION")
print("=" * 70)
print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")


## 1. Load data and baselines


In [ ]:
print("\n" + "=" * 70)
print("1. Loading data and baseline models")
print("=" * 70)

ell = np.load(os.path.join(PROCESSED_PATH, 'elliptic_splits.npz'))
eth = np.load(os.path.join(PROCESSED_PATH, 'ethereum_splits.npz'))
X_train_ell, y_train_ell = ell['X_train'], ell['y_train']
X_test_ell, y_test_ell = ell['X_test'], ell['y_test']
X_train_eth, y_train_eth = eth['X_train'], eth['y_train']
X_test_eth, y_test_eth = eth['X_test'], eth['y_test']

eval_idx_ell = np.load(os.path.join(RESULTS_PATH, 'xai_eval_indices_elliptic.npy'))[:N_EVAL]
eval_idx_eth = np.load(os.path.join(RESULTS_PATH, 'xai_eval_indices_ethereum.npy'))[:N_EVAL]

rf_ell_o = joblib.load(os.path.join(MODELS_PATH, 'elliptic_rf.joblib'))
lgb_ell_o = joblib.load(os.path.join(MODELS_PATH, 'elliptic_lgb.joblib'))
rf_eth_o = joblib.load(os.path.join(MODELS_PATH, 'ethereum_rf.joblib'))
lgb_eth_o = joblib.load(os.path.join(MODELS_PATH, 'ethereum_lgb.joblib'))

print(f"  Elliptic  train={len(y_train_ell)}  fraud={y_train_ell.mean():.2%}")
print(f"  Ethereum  train={len(y_train_eth)}  fraud={y_train_eth.mean():.2%}")


## 2. Train cost-sensitive (balanced) models

RF uses `class_weight='balanced'`; LightGBM uses
`scale_pos_weight = n_negative / n_positive`.


In [ ]:
print("\n" + "=" * 70)
print("2. Training cost-sensitive models")
print("=" * 70)

rf_ell_b = RandomForestClassifier(
    n_estimators=300, max_depth=20, min_samples_split=5, min_samples_leaf=2,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1,
).fit(X_train_ell, y_train_ell)
joblib.dump(rf_ell_b, os.path.join(MODELS_PATH, 'elliptic_rf_balanced.joblib'))
print("  RF Elliptic (balanced) saved")

rf_eth_b = RandomForestClassifier(
    n_estimators=300, max_depth=20, min_samples_split=5, min_samples_leaf=2,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1,
).fit(X_train_eth, y_train_eth)
joblib.dump(rf_eth_b, os.path.join(MODELS_PATH, 'ethereum_rf_balanced.joblib'))
print("  RF Ethereum (balanced) saved")

spw_ell = float((y_train_ell == 0).sum() / max((y_train_ell == 1).sum(), 1))
spw_eth = float((y_train_eth == 0).sum() / max((y_train_eth == 1).sum(), 1))
lgb_ell_b = lgb.LGBMClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.05, num_leaves=31,
    scale_pos_weight=spw_ell, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
).fit(X_train_ell, y_train_ell)
joblib.dump(lgb_ell_b, os.path.join(MODELS_PATH, 'elliptic_lgb_balanced.joblib'))
print(f"  LGB Elliptic (scale_pos_weight={spw_ell:.2f}) saved")

lgb_eth_b = lgb.LGBMClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.05, num_leaves=31,
    scale_pos_weight=spw_eth, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
).fit(X_train_eth, y_train_eth)
joblib.dump(lgb_eth_b, os.path.join(MODELS_PATH, 'ethereum_lgb_balanced.joblib'))
print(f"  LGB Ethereum (scale_pos_weight={spw_eth:.2f}) saved")


## 3. Predictive performance: Original vs Balanced


In [ ]:
print("\n" + "=" * 70)
print("3. Predictive performance")
print("=" * 70)

def perf(model, X, y, name, dataset, mode):
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    row = {
        'Model': name, 'Dataset': dataset, 'Mode': mode,
        'F1': float(f1_score(y, y_pred)),
        'Precision': float(precision_score(y, y_pred, zero_division=0)),
        'Recall': float(recall_score(y, y_pred, zero_division=0)),
        'AUC': float(roc_auc_score(y, y_proba)),
    }
    print(f"  {name}-{dataset} ({mode}): F1={row['F1']:.4f} "
          f"P={row['Precision']:.4f} R={row['Recall']:.4f} AUC={row['AUC']:.4f}")
    return row

perf_rows = [
    perf(rf_ell_o, X_test_ell, y_test_ell, 'RF',  'Elliptic', 'Original'),
    perf(lgb_ell_o, X_test_ell, y_test_ell, 'LGB', 'Elliptic', 'Original'),
    perf(rf_eth_o, X_test_eth, y_test_eth, 'RF',  'Ethereum', 'Original'),
    perf(lgb_eth_o, X_test_eth, y_test_eth, 'LGB', 'Ethereum', 'Original'),
    perf(rf_ell_b, X_test_ell, y_test_ell, 'RF',  'Elliptic', 'Balanced'),
    perf(lgb_ell_b, X_test_ell, y_test_ell, 'LGB', 'Elliptic', 'Balanced'),
    perf(rf_eth_b, X_test_eth, y_test_eth, 'RF',  'Ethereum', 'Balanced'),
    perf(lgb_eth_b, X_test_eth, y_test_eth, 'LGB', 'Ethereum', 'Balanced'),
]
perf_df = pd.DataFrame(perf_rows)


## 4. Compute SHAP and LIME attributions for the balanced models


In [ ]:
print("\n" + "=" * 70)
print("4. SHAP and LIME on balanced models")
print("=" * 70)

shap_exp_rf_ell_b  = ShapTreeExplainer(rf_ell_b)
shap_exp_lgb_ell_b = ShapTreeExplainer(lgb_ell_b)
shap_exp_rf_eth_b  = ShapTreeExplainer(rf_eth_b)
shap_exp_lgb_eth_b = ShapTreeExplainer(lgb_eth_b)

def shap_on_subset(explainer, X_test, indices, label):
    sv = explainer.explain(X_test, indices=indices).astype(np.float32)
    print(f"  SHAP {label}: shape={sv.shape}")
    return sv

shap_rf_ell_b  = shap_on_subset(shap_exp_rf_ell_b,  X_test_ell, eval_idx_ell, 'RF-Elliptic-Balanced')
shap_lgb_ell_b = shap_on_subset(shap_exp_lgb_ell_b, X_test_ell, eval_idx_ell, 'LGB-Elliptic-Balanced')
shap_rf_eth_b  = shap_on_subset(shap_exp_rf_eth_b,  X_test_eth, eval_idx_eth, 'RF-Ethereum-Balanced')
shap_lgb_eth_b = shap_on_subset(shap_exp_lgb_eth_b, X_test_eth, eval_idx_eth, 'LGB-Ethereum-Balanced')

np.save(os.path.join(RESULTS_PATH, 'exp12_balanced_shap_rf_elliptic.npy'), shap_rf_ell_b)
np.save(os.path.join(RESULTS_PATH, 'exp12_balanced_shap_lgb_elliptic.npy'), shap_lgb_ell_b)
np.save(os.path.join(RESULTS_PATH, 'exp12_balanced_shap_rf_ethereum.npy'), shap_rf_eth_b)
np.save(os.path.join(RESULTS_PATH, 'exp12_balanced_shap_lgb_ethereum.npy'), shap_lgb_eth_b)

lime_exp_ell = LimeTabularExplainer(X_train_ell, random_state=42)
lime_exp_eth = LimeTabularExplainer(X_train_eth, random_state=42)

def proba_fn(model):
    base = make_fraud_predict_fn(model, X_train_ell.shape[1])
    def _fn(X):
        p = make_fraud_predict_fn(model, X.shape[1])(X)
        return np.column_stack([1 - p, p])
    return _fn

def lime_on_subset(lime_exp, model, X_test, indices, label):
    attrs = lime_exp.explain(
        X_test, predict_proba=proba_fn(model),
        indices=indices[:N_EVAL_LIME], verbose_every=25, name=label,
    ).astype(np.float32)
    print(f"  LIME {label}: shape={attrs.shape}")
    return attrs

lime_rf_ell_b  = lime_on_subset(lime_exp_ell, rf_ell_b,  X_test_ell, eval_idx_ell, 'RF-Elliptic-Balanced')
lime_lgb_ell_b = lime_on_subset(lime_exp_ell, lgb_ell_b, X_test_ell, eval_idx_ell, 'LGB-Elliptic-Balanced')
lime_rf_eth_b  = lime_on_subset(lime_exp_eth, rf_eth_b,  X_test_eth, eval_idx_eth, 'RF-Ethereum-Balanced')
lime_lgb_eth_b = lime_on_subset(lime_exp_eth, lgb_eth_b, X_test_eth, eval_idx_eth, 'LGB-Ethereum-Balanced')

np.save(os.path.join(RESULTS_PATH, 'exp12_balanced_lime_rf_elliptic.npy'), lime_rf_ell_b)
np.save(os.path.join(RESULTS_PATH, 'exp12_balanced_lime_lgb_elliptic.npy'), lime_lgb_ell_b)
np.save(os.path.join(RESULTS_PATH, 'exp12_balanced_lime_rf_ethereum.npy'), lime_rf_eth_b)
np.save(os.path.join(RESULTS_PATH, 'exp12_balanced_lime_lgb_ethereum.npy'), lime_lgb_eth_b)


## 5. Re-run Modules 1-3 on both modes

Load the Original attributions from 03a and run the full set of metrics
(M1: Comp@5, Suff@5, Infidelity; M2: Lipschitz, Kendall τ, CoV, Identity;
M3: RAS, DVR, BRAS) for both Original and Balanced.


In [ ]:
shap_rf_ell_o  = np.load(os.path.join(RESULTS_PATH, 'shap_values_rf_elliptic.npy'))
shap_lgb_ell_o = np.load(os.path.join(RESULTS_PATH, 'shap_values_lgb_elliptic.npy'))
shap_rf_eth_o  = np.load(os.path.join(RESULTS_PATH, 'shap_values_rf_ethereum.npy'))
shap_lgb_eth_o = np.load(os.path.join(RESULTS_PATH, 'shap_values_lgb_ethereum.npy'))
lime_rf_ell_o  = np.load(os.path.join(RESULTS_PATH, 'lime_attrs_rf_elliptic.npy'))
lime_lgb_ell_o = np.load(os.path.join(RESULTS_PATH, 'lime_attrs_lgb_elliptic.npy'))
lime_rf_eth_o  = np.load(os.path.join(RESULTS_PATH, 'lime_attrs_rf_ethereum.npy'))
lime_lgb_eth_o = np.load(os.path.join(RESULTS_PATH, 'lime_attrs_lgb_ethereum.npy'))


def eval_config(
    model_orig, model_bal, X, y, indices, attrs_orig, attrs_bal,
    model_name, dataset, xai_name, lime_exp=None, shap_exp_bal=None,
):
    rule_fn = elliptic_rules if dataset == 'Elliptic' else ethereum_rules
    # Truncate all arrays to the shortest common length
    n_eval = min(len(indices), len(attrs_orig), len(attrs_bal))
    indices = indices[:n_eval]
    attrs_orig = attrs_orig[:n_eval]
    attrs_bal = attrs_bal[:n_eval]

    rows = []
    for mode, model, attrs in [('Original', model_orig, attrs_orig),
                                ('Balanced', model_bal,  attrs_bal)]:
        predict_fn = make_fraud_predict_fn(model, X.shape[1])
        rng = np.random.default_rng(42)

        # Module 1
        comp = comprehensiveness(predict_fn, X, attrs, indices, k=K_FID)
        suff = sufficiency(predict_fn, X, attrs, indices, k=K_FID)
        infid = infidelity(predict_fn, X, attrs, indices,
                           n_perturbations=N_PERTURBATIONS, sigma=SIGMA, rng=rng)

        # Module 2 (Lipschitz, Kendall from cached attrs; CoV/Identity from fresh runs)
        lip = lipschitz_stability(X, attrs, indices)
        kendall = rank_stability_kendall(attrs)
        if xai_name == 'SHAP':
            fn_cov = (shap_exp_bal if mode == 'Balanced' else ShapTreeExplainer(model)).make_explain_fn()
        else:
            fn_cov = lime_exp.make_explain_fn(
                lambda X: np.column_stack([1 - predict_fn(X), predict_fn(X)])
            )
        cov_vals, id_vals = [], []
        rng_m2 = np.random.default_rng(42)
        for i in range(min(N_COV_SAMPLE, n_eval)):
            x = X[indices[i]]
            cov_vals.append(cov_bootstrap(fn_cov, x, n_bootstrap=N_BOOTSTRAP,
                                          perturbation_scale=PERT_SCALE, rng=rng_m2))
            id_vals.append(identity_score(fn_cov, x, n_runs=N_IDENTITY))
        cov_mean = float(np.mean(cov_vals)) if cov_vals else 0.0
        identity_mean = float(np.mean(id_vals)) if id_vals else 0.0

        # Module 3 on fraud subset
        fraud_mask = y[indices] == 1
        if fraud_mask.any():
            bras_out = evaluate_bras(X, attrs[fraud_mask], indices[fraud_mask],
                                     rule_fn=rule_fn, k=TOP_K_BRAS, alpha=ALPHA_BRAS)
        else:
            bras_out = {'RAS': 0.0, 'DVR': 0.0, 'BRAS': 0.0, 'N_eval': 0}

        row = {
            'Model': model_name, 'Dataset': dataset, 'XAI': xai_name, 'Mode': mode,
            'Comp@5': comp, 'Suff@5': suff, 'Infidelity': infid,
            'Lipschitz': lip, 'Kendall_tau': kendall,
            'CoV_Bootstrap': cov_mean, 'Identity': identity_mean,
            'RAS': bras_out['RAS'], 'DVR': bras_out['DVR'], 'BRAS': bras_out['BRAS'],
        }
        print(f"  {model_name}-{dataset}-{xai_name}-{mode}: "
              f"Comp@5={comp:.3f}  Suff@5={suff:.3f}  Lip={lip:.3f}  BRAS={bras_out['BRAS']:.3f}")
        rows.append(row)
    return rows

all_rows: list[dict] = []
CONFIGS = [
    ('RF',  'Elliptic', 'SHAP', rf_ell_o,  rf_ell_b,  X_test_ell, y_test_ell, eval_idx_ell, shap_rf_ell_o,  shap_rf_ell_b,  None,          shap_exp_rf_ell_b),
    ('LGB', 'Elliptic', 'SHAP', lgb_ell_o, lgb_ell_b, X_test_ell, y_test_ell, eval_idx_ell, shap_lgb_ell_o, shap_lgb_ell_b, None,          shap_exp_lgb_ell_b),
    ('RF',  'Ethereum', 'SHAP', rf_eth_o,  rf_eth_b,  X_test_eth, y_test_eth, eval_idx_eth, shap_rf_eth_o,  shap_rf_eth_b,  None,          shap_exp_rf_eth_b),
    ('LGB', 'Ethereum', 'SHAP', lgb_eth_o, lgb_eth_b, X_test_eth, y_test_eth, eval_idx_eth, shap_lgb_eth_o, shap_lgb_eth_b, None,          shap_exp_lgb_eth_b),
    ('RF',  'Elliptic', 'LIME', rf_ell_o,  rf_ell_b,  X_test_ell, y_test_ell, eval_idx_ell[:N_EVAL_LIME], lime_rf_ell_o,  lime_rf_ell_b,  lime_exp_ell,  None),
    ('LGB', 'Elliptic', 'LIME', lgb_ell_o, lgb_ell_b, X_test_ell, y_test_ell, eval_idx_ell[:N_EVAL_LIME], lime_lgb_ell_o, lime_lgb_ell_b, lime_exp_ell,  None),
    ('RF',  'Ethereum', 'LIME', rf_eth_o,  rf_eth_b,  X_test_eth, y_test_eth, eval_idx_eth[:N_EVAL_LIME], lime_rf_eth_o,  lime_rf_eth_b,  lime_exp_eth,  None),
    ('LGB', 'Ethereum', 'LIME', lgb_eth_o, lgb_eth_b, X_test_eth, y_test_eth, eval_idx_eth[:N_EVAL_LIME], lime_lgb_eth_o, lime_lgb_eth_b, lime_exp_eth,  None),
]
for (model_name, dataset, xai, mo, mb, X, y, idx, ao, ab, lex, sexp) in CONFIGS:
    print(f"\n--- {model_name}-{dataset}-{xai} ---")
    all_rows.extend(eval_config(
        mo, mb, X, y, idx, ao, ab,
        model_name=model_name, dataset=dataset, xai_name=xai,
        lime_exp=lex, shap_exp_bal=sexp,
    ))

results_df = pd.DataFrame(all_rows)


## 6. Deltas (Balanced  Original)


In [ ]:
metric_cols = ['Comp@5', 'Suff@5', 'Infidelity', 'Lipschitz', 'Kendall_tau',
               'CoV_Bootstrap', 'Identity', 'RAS', 'DVR', 'BRAS']
orig_df = results_df[results_df['Mode'] == 'Original'].reset_index(drop=True)
bal_df  = results_df[results_df['Mode'] == 'Balanced'].reset_index(drop=True)

comparison_rows = []
for i in range(len(orig_df)):
    row = {
        'Model': orig_df.iloc[i]['Model'],
        'Dataset': orig_df.iloc[i]['Dataset'],
        'XAI': orig_df.iloc[i]['XAI'],
    }
    for col in metric_cols:
        row[f'{col}_Orig'] = orig_df.iloc[i][col]
        row[f'{col}_Bal']  = bal_df.iloc[i][col]
        row[f'{col}_Delta'] = bal_df.iloc[i][col] - orig_df.iloc[i][col]
    comparison_rows.append(row)
comparison_df = pd.DataFrame(comparison_rows)
delta_cols = [c for c in comparison_df.columns if c.endswith('_Delta')]
print(comparison_df[['Model', 'Dataset', 'XAI'] + delta_cols].to_string(index=False))


## 7. Visualizations


In [ ]:
print("\n" + "=" * 70)
print("7. Visualizations")
print("=" * 70)

COLORS_MODE = {'Original': '#3498db', 'Balanced': '#e67e22'}

def _grouped(metrics, title, out_name, ylim=None):
    configs = results_df[results_df['Mode'] == 'Original'][['Model', 'Dataset', 'XAI']].values
    labels = [f"{c[0]}\n{c[2]}\n{c[1]}" for c in configs]
    n = len(labels)
    fig, axes = plt.subplots(1, len(metrics), figsize=(6 * len(metrics), 7), squeeze=False)
    for ax_idx, metric in enumerate(metrics):
        ax = axes[0, ax_idx]
        o_vals = results_df[results_df['Mode'] == 'Original'][metric].values
        b_vals = results_df[results_df['Mode'] == 'Balanced'][metric].values
        x = np.arange(n); w = 0.35
        ax.bar(x - w / 2, o_vals, w, label='Original', color=COLORS_MODE['Original'],
               alpha=0.85, edgecolor='black', linewidth=0.5)
        ax.bar(x + w / 2, b_vals, w, label='Balanced', color=COLORS_MODE['Balanced'],
               alpha=0.85, edgecolor='black', linewidth=0.5)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=7, rotation=45, ha='right')
        ax.set_ylabel(metric, fontsize=10)
        ax.set_title(metric, fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')
        if ylim is not None:
            ax.set_ylim(*ylim)
        ax.legend(fontsize=8)
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_PATH, out_name), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  wrote {out_name}")


_grouped(['Comp@5', 'Suff@5', 'Infidelity'],
         'Exp 1.2  Fidelity  Original vs Balanced',
         'exp12_faithfulness_comparison.png')
_grouped(['Lipschitz', 'Kendall_tau', 'Identity'],
         'Exp 1.2  Stability  Original vs Balanced',
         'exp12_stability_comparison.png')
_grouped(['RAS', 'DVR', 'BRAS'],
         'Exp 1.2  BRAS  Original vs Balanced',
         'exp12_bras_comparison.png')

# Delta heatmap
higher_better = {'Comp@5': True, 'Suff@5': True, 'Infidelity': False,
                 'Lipschitz': False, 'Kendall_tau': True, 'CoV_Bootstrap': False,
                 'Identity': True, 'RAS': True, 'DVR': False, 'BRAS': True}
delta_arr = np.zeros((len(comparison_df), len(metric_cols)))
for j, m in enumerate(metric_cols):
    d = comparison_df[f'{m}_Delta'].values
    delta_arr[:, j] = d if higher_better[m] else -d

vmax = max(np.percentile(np.abs(delta_arr[np.isfinite(delta_arr)]), 95), 0.01)
fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(delta_arr, cmap=plt.cm.RdYlGn,
               norm=mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax), aspect='auto')
config_labels = [f"{r['Model']}-{r['XAI']}-{r['Dataset']}" for _, r in comparison_df.iterrows()]
ax.set_xticks(range(len(metric_cols))); ax.set_xticklabels(metric_cols, fontsize=9, rotation=45, ha='right')
ax.set_yticks(range(len(config_labels))); ax.set_yticklabels(config_labels, fontsize=9)
for i in range(delta_arr.shape[0]):
    for j in range(delta_arr.shape[1]):
        raw = comparison_df.iloc[i][f'{metric_cols[j]}_Delta']
        color = 'white' if abs(delta_arr[i, j]) > vmax * 0.6 else 'black'
        ax.text(j, i, f'{raw:+.3f}', ha='center', va='center', fontsize=7, color=color)
ax.set_title('Exp 1.2  Deltas (Balanced - Original)\nGreen = improvement, Red = degradation',
             fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, label='Impact (positive = improvement)')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'exp12_delta_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print("  wrote exp12_delta_heatmap.png")

# Performance comparison
fig, axes = plt.subplots(1, 4, figsize=(20, 6))
for ax_idx, metric in enumerate(['F1', 'Precision', 'Recall', 'AUC']):
    ax = axes[ax_idx]
    configs = perf_df[perf_df['Mode'] == 'Original'][['Model', 'Dataset']].values
    labels = [f"{c[0]}\n{c[1]}" for c in configs]
    n = len(labels)
    x = np.arange(n); w = 0.35
    o_vals = perf_df[perf_df['Mode'] == 'Original'][metric].values
    b_vals = perf_df[perf_df['Mode'] == 'Balanced'][metric].values
    ax.bar(x - w / 2, o_vals, w, label='Original', color=COLORS_MODE['Original'],
           alpha=0.85, edgecolor='black', linewidth=0.5)
    ax.bar(x + w / 2, b_vals, w, label='Balanced', color=COLORS_MODE['Balanced'],
           alpha=0.85, edgecolor='black', linewidth=0.5)
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8, rotation=45, ha='right')
    ax.set_ylabel(metric, fontsize=10); ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y'); ax.set_ylim(0, 1.05); ax.legend(fontsize=8)
fig.suptitle('Exp 1.2  Predictive performance  Original vs Balanced',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'exp12_performance_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print("  wrote exp12_performance_comparison.png")


## 8. Save results


In [ ]:
results_df.to_csv(os.path.join(RESULTS_PATH, 'exp12_imbalance_results.csv'), index=False)
comparison_df.to_csv(os.path.join(RESULTS_PATH, 'exp12_imbalance_comparison.csv'), index=False)
perf_df.to_csv(os.path.join(RESULTS_PATH, 'exp12_performance_results.csv'), index=False)

print("""
Files saved:
  exp12_imbalance_results.csv     (Module 1-3 metrics x 2 modes x 8 configs)
  exp12_imbalance_comparison.csv  (Orig / Bal / Delta per metric)
  exp12_performance_results.csv   (F1, P, R, AUC for both modes)
  exp12_{faithfulness,stability,bras,delta,performance}_*.png

Balanced models saved:
  elliptic_{rf,lgb}_balanced.joblib
  ethereum_{rf,lgb}_balanced.joblib
Balanced XAI cached:
  exp12_balanced_{shap,lime}_{rf,lgb}_{elliptic,ethereum}.npy
""")
